[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/your-org/pypath/blob/main/notebooks/module5/02-sklearn-basics.ipynb)

# Module 5 — Lesson 2: Scikit-Learn Basics

**Module:** 5 — Machine Learning Foundations | **Time:** 25 minutes

## Learning Objectives

By the end of this lesson you will be able to:

- Describe the consistent `fit / transform / predict` API pattern used throughout scikit-learn
- Apply the major preprocessing transformers: scalers, encoders, and imputers
- Build preprocessing and modelling `Pipeline` objects
- Use `ColumnTransformer` to handle mixed numeric and categorical feature sets
- Run hyperparameter searches with `GridSearchCV` and `RandomizedSearchCV`

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.datasets import load_breast_cancer, make_classification
from sklearn.model_selection import train_test_split, GridSearchCV, RandomizedSearchCV, cross_val_score
from sklearn.preprocessing import (StandardScaler, MinMaxScaler, RobustScaler,
                                   LabelEncoder, OrdinalEncoder, OneHotEncoder)
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.metrics import accuracy_score, classification_report

np.random.seed(42)
plt.rcParams.update({'figure.dpi': 110, 'font.size': 11})
print('All imports successful.')

## 1. The Fit / Transform / Predict API

Every scikit-learn object follows one consistent contract:

| Object type | Key methods | When to call |
|---|---|---|
| **Transformer** | `fit(X)`, `transform(X)`, `fit_transform(X)` | Preprocessing steps |
| **Estimator (Classifier / Regressor)** | `fit(X, y)`, `predict(X)`, `predict_proba(X)` | Modelling step |
| **Pipeline** | `fit(X, y)`, `predict(X)` | Chains transformers + estimator |

The golden rule: **fit only on training data**, then transform both train and test. Never leak test statistics into training.

In [ ]:
# Demonstrate the fit/transform contract
X_demo = np.array([[1.0, 200.0], [2.0, 400.0], [3.0, 600.0],
                   [100.0, 800.0], [5.0, 1000.0]])

scaler = StandardScaler()

# fit computes mean and std from training data
scaler.fit(X_demo[:4])  # fit on first 4 rows (simulated train set)
print('Learned mean:  ', scaler.mean_.round(2))
print('Learned scale: ', scaler.scale_.round(2))

# transform applies the stored statistics
X_scaled_train = scaler.transform(X_demo[:4])
X_scaled_test  = scaler.transform(X_demo[4:])  # uses TRAIN statistics — no leakage

print('\nScaled train:\n', X_scaled_train.round(3))
print('\nScaled test (using train mean/std):\n', X_scaled_test.round(3))
print('\nfit_transform is equivalent to fit then transform on the SAME data.')

## 2. Scalers — StandardScaler, MinMaxScaler, RobustScaler

- **StandardScaler** — zero mean, unit variance. Assumes approximate normality.
- **MinMaxScaler** — maps values to `[0, 1]`. Sensitive to outliers.
- **RobustScaler** — uses median and IQR. Resistant to outliers.

In [ ]:
# Compare scalers on data with an outlier
np.random.seed(0)
X_vals = np.concatenate([np.random.normal(5, 1, 99), [50.0]]).reshape(-1, 1)  # one outlier at 50

scalers = {
    'StandardScaler': StandardScaler(),
    'MinMaxScaler':   MinMaxScaler(),
    'RobustScaler':   RobustScaler()
}

fig, axes = plt.subplots(1, 3, figsize=(14, 4))
for ax, (name, sc) in zip(axes, scalers.items()):
    X_sc = sc.fit_transform(X_vals)
    ax.hist(X_sc[:-1], bins=20, color='steelblue', alpha=0.7, label='Normal data')
    ax.axvline(X_sc[-1], color='red', lw=2, linestyle='--', label=f'Outlier → {X_sc[-1][0]:.2f}')
    ax.set_title(name, fontweight='bold')
    ax.set_xlabel('Scaled value')
    ax.legend(fontsize=8)

plt.suptitle('Effect of Scalers on Data with Outlier', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()
print('RobustScaler confines the outlier far less than StandardScaler and MinMaxScaler do.')

## 3. Encoders — LabelEncoder, OrdinalEncoder, OneHotEncoder

| Encoder | Input | Output | Use case |
|---|---|---|---|
| `LabelEncoder` | 1-D target array | Integer labels | Encoding `y` only |
| `OrdinalEncoder` | 2-D feature matrix | Integer-coded columns | Ordered categories |
| `OneHotEncoder` | 2-D feature matrix | Binary columns | Nominal categories |

OneHotEncoding avoids imposing false ordinal relationships between categories.

In [ ]:
# Encoding demonstration
categories_data = np.array([['red', 'small'],
                            ['blue', 'medium'],
                            ['green', 'large'],
                            ['red', 'medium'],
                            ['blue', 'small']])

print('--- OrdinalEncoder ---')
ord_enc = OrdinalEncoder()
print(ord_enc.fit_transform(categories_data))

print('\n--- OneHotEncoder (sparse=False) ---')
ohe = OneHotEncoder(sparse_output=False)
ohe_result = ohe.fit_transform(categories_data)
print('Feature names:', ohe.get_feature_names_out())
print(ohe_result)

print('\n--- LabelEncoder (for target y) ---')
le = LabelEncoder()
y_str = ['cat', 'dog', 'bird', 'dog', 'cat']
print('Original:', y_str)
print('Encoded: ', le.fit_transform(y_str))
print('Classes: ', le.classes_)

## 4. SimpleImputer — Handling Missing Values

`SimpleImputer` fills `NaN` values with a chosen strategy before passing data to a model. Always fit on training data only.

In [ ]:
# Create a small dataset with missing values
df_missing = pd.DataFrame({
    'age':    [25.0, np.nan, 35.0, 42.0, np.nan, 55.0],
    'income': [40000, 60000, np.nan, 80000, 55000, np.nan],
    'city':   ['NY', 'LA', np.nan, 'NY', 'LA', 'NY']
})
print('Before imputation:')
print(df_missing)
print(f'Missing values:\n{df_missing.isnull().sum()}')

# Numeric: mean imputation
num_imputer = SimpleImputer(strategy='mean')
df_num_imp = df_missing[['age', 'income']].copy()
df_num_imp[:] = num_imputer.fit_transform(df_num_imp)

# Categorical: most_frequent
cat_imputer = SimpleImputer(strategy='most_frequent')
df_cat_imp = df_missing[['city']].copy()
df_cat_imp[:] = cat_imputer.fit_transform(df_cat_imp)

df_fixed = pd.concat([df_num_imp, df_cat_imp], axis=1)
print('\nAfter imputation:')
print(df_fixed)

## 5. Pipeline — Chaining Steps

A `Pipeline` packages transformers and an estimator into a single object. Benefits:

1. Prevents data leakage — `fit` on train propagates correctly through all steps
2. One object to cross-validate, serialize, and deploy
3. Code is cleaner and reproducible

In [ ]:
data = load_breast_cancer()
X, y = data.data, data.target
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, stratify=y, random_state=42)

# Pipeline: impute -> scale -> classify
pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler',  StandardScaler()),
    ('clf',     LogisticRegression(max_iter=1000))
])

pipeline.fit(X_train, y_train)
y_pred = pipeline.predict(X_test)

print('Pipeline steps:', [name for name, _ in pipeline.steps])
print(f'\nTest Accuracy: {accuracy_score(y_test, y_pred):.4f}')
print('\nClassification Report:')
print(classification_report(y_test, y_pred, target_names=data.target_names))

## 6. ColumnTransformer — Mixed Feature Types

Real datasets combine numeric and categorical columns. `ColumnTransformer` applies different transformers to different subsets of columns, then concatenates the results.

In [ ]:
# Simulate a Titanic-like dataset
np.random.seed(42)
n = 500
df = pd.DataFrame({
    'age':    np.random.normal(30, 12, n).clip(1, 80),
    'fare':   np.random.exponential(30, n),
    'pclass': np.random.choice([1, 2, 3], n),
    'sex':    np.random.choice(['male', 'female'], n),
    'embarked': np.random.choice(['S', 'C', 'Q'], n)
})
# Introduce some missing values
df.loc[np.random.choice(n, 30, replace=False), 'age'] = np.nan
df.loc[np.random.choice(n, 10, replace=False), 'embarked'] = np.nan
# Synthetic target
y_syn = ((df['pclass'] == 1).astype(int) +
         (df['sex'] == 'female').astype(int) +
         (df['fare'] > 50).astype(int)).clip(0, 1).values

X_syn = df
X_tr, X_te, y_tr, y_te = train_test_split(X_syn, y_syn, test_size=0.2, random_state=42)

numeric_features     = ['age', 'fare']
categorical_features = ['pclass', 'sex', 'embarked']

numeric_transformer = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler',  StandardScaler())
])

categorical_transformer = Pipeline([
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('ohe',     OneHotEncoder(handle_unknown='ignore', sparse_output=False))
])

preprocessor = ColumnTransformer([
    ('num', numeric_transformer,     numeric_features),
    ('cat', categorical_transformer, categorical_features)
])

full_pipe = Pipeline([
    ('preprocessor', preprocessor),
    ('clf', RandomForestClassifier(n_estimators=100, random_state=42))
])

full_pipe.fit(X_tr, y_tr)
acc = accuracy_score(y_te, full_pipe.predict(X_te))
print('ColumnTransformer + RandomForest accuracy:', round(acc, 4))
print('Transformed feature count:', preprocessor.fit_transform(X_tr, y_tr).shape[1])

## 7. GridSearchCV and RandomizedSearchCV

- **`GridSearchCV`** — exhaustively tries every combination; guarantees finding the best within the grid
- **`RandomizedSearchCV`** — samples `n_iter` random combinations; faster for large search spaces

Both integrate with `Pipeline` via the `step__param` naming convention.

In [ ]:
from scipy.stats import randint, uniform

# GridSearchCV on the full pipeline
param_grid = {
    'clf__n_estimators': [50, 100],
    'clf__max_depth':    [3, 5, None]
}

grid_search = GridSearchCV(
    full_pipe, param_grid,
    cv=3, scoring='accuracy', n_jobs=-1, verbose=0
)
grid_search.fit(X_tr, y_tr)

print('=== GridSearchCV ===')
print('Best params:   ', grid_search.best_params_)
print('Best CV score: ', round(grid_search.best_score_, 4))
print('Test accuracy: ', round(accuracy_score(y_te, grid_search.best_estimator_.predict(X_te)), 4))

# RandomizedSearchCV
param_dist = {
    'clf__n_estimators': randint(50, 300),
    'clf__max_depth':    [3, 5, 7, None],
    'clf__min_samples_split': randint(2, 20)
}

rand_search = RandomizedSearchCV(
    full_pipe, param_dist,
    n_iter=15, cv=3, scoring='accuracy', random_state=42, n_jobs=-1
)
rand_search.fit(X_tr, y_tr)

print('\n=== RandomizedSearchCV (15 iterations) ===')
print('Best params:   ', rand_search.best_params_)
print('Best CV score: ', round(rand_search.best_score_, 4))
print('Test accuracy: ', round(accuracy_score(y_te, rand_search.best_estimator_.predict(X_te)), 4))

## 8. Search Results Visualisation

In [ ]:
results_df = pd.DataFrame(grid_search.cv_results_)
results_df = results_df[['param_clf__n_estimators', 'param_clf__max_depth',
                          'mean_test_score', 'std_test_score']].sort_values('mean_test_score', ascending=False)
results_df.columns = ['n_estimators', 'max_depth', 'mean_cv_acc', 'std_cv_acc']
results_df = results_df.reset_index(drop=True)
print('GridSearchCV Results:')
print(results_df.round(4).to_string())

plt.figure(figsize=(9, 4))
plt.barh(range(len(results_df)), results_df['mean_cv_acc'],
         xerr=results_df['std_cv_acc'], color='steelblue', alpha=0.8)
plt.yticks(range(len(results_df)),
           [f"n={r.n_estimators}, d={r.max_depth}" for _, r in results_df.iterrows()],
           fontsize=9)
plt.xlabel('Mean CV Accuracy')
plt.title('GridSearchCV Results')
plt.tight_layout()
plt.show()

## Practice Exercises

**Exercise 1 — Scaler Comparison on Real Data**
Load `load_breast_cancer()`. Create three pipelines each using a different scaler (`StandardScaler`, `MinMaxScaler`, `RobustScaler`) followed by `LogisticRegression`. Use 5-fold cross-validation to compare mean accuracy across scalers. Print a summary table.

**Exercise 2 — ColumnTransformer Extension**
Extend the Titanic-like dataset by adding an `'age_missing'` binary indicator column (1 where age was NaN, 0 otherwise) before fitting. Modify the `ColumnTransformer` to include this new column as a passthrough feature. Check if accuracy improves.

**Exercise 3 — Hyperparameter Search Visualisation**
Run `RandomizedSearchCV` with 30 iterations on a `Pipeline(StandardScaler, LogisticRegression)` over `logisticregression__C` values from `loguniform(1e-4, 1e2)`. Plot CV score vs log(C) as a scatter plot and draw a vertical line at the best C found.